## Workshop: Fine-tuning TinyDNABERT for DNA sequence classification tasks
Model: https://huggingface.co/ChengsenWang/TinyDNABERT-20M-V1

Dataset: https://huggingface.co/datasets/InstaDeepAI/nucleotide_transformer_downstream_tasks_revised

1. Load necessary packages

In [ ]:
import torch
from typing import Any, Tuple, Union
import tensorflow as tf
import numpy as np
import pandas as pd
import scipy as sp

from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForMaskedLM,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
    DataCollatorForLanguageModeling,
    EvalPrediction,
)
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from scipy.special import softmax

from peft import get_peft_model, LoraConfig, TaskType

2. Load the dataset

In [ ]:
dataset = # YOUR CODE HERE
print(dataset)

  a. Separate the train and test datasets.

In [ ]:
# Train dataset
train_dataset = # YOUR CODE HERE

# Evaluation dataset
test_dataset = # YOUR CODE HERE

  b. Filter only for the promoter classification task. This is a binary classification task which involves categorising each sample as "promoter" (1) or "not promoter" (0).

  *Hint: use the `filter` function*

In [ ]:
train_dataset = # YOUR CODE HERE
test_dataset = # YOUR CODE HERE

Filter:   0%|          | 0/493242 [00:00<?, ? examples/s]

b. Split the train dataset further into train (80%) and validation (20%).

*Hint: use the `train_test_split` function.*

In [ ]:
# Split the training data into training and validation
split_dataset = # YOUR CODE HERE
print(split_dataset)

In [ ]:
train_dataset = # YOUR CODE HERE
val_dataset = # YOUR CODE HERE

In [ ]:
print(f"No. train samples: {len(train_dataset)}\nNo. val. samples: {len(val_dataset)}\nNo. test samples: {len(test_dataset)}")

No. train samples: 24000
No. val. samples: 6000
No. test samples: 1584


d. To make the training less time-consuming, we will use only the first 10000 rows of the training data.

*Hint: use the `select` function.*

In [ ]:
train_dataset = # YOUR CODE HERE

3. Load the tokeniser and define the `preprocess_function`, which we will use to batch apply tokenisation to our datasets.

In [ ]:
# Set the model name from the Huggingface page
MODEL_NAME =  # YOUR CODE HERE

In [ ]:
# Load the tokeniser
tokeniser =  # YOUR CODE HERE

In [ ]:
# Define the preprocessing function
def preprocess_function(examples):
    return tokeniser(
        examples["sequence"],
        truncation=True,
        # YOUR CODE HERE: SET THE MAXIMUM LENGTH AS 512
        # YOUR CODE HERE: PAD THE TOKENISED SEQUENCE TO THE MAXIMUM LENGTH
    )

4. Tokenise the data

*Hint: use the `map` function*

In [ ]:
train_ds_encoded = # YOUR CODE HERE
val_ds_encoded = # YOUR CODE HERE
test_ds_encoded = # YOUR CODE HERE

5. Fine-tune for the promoter classification task

In [ ]:
# Load the pretrained weights and add a classification head
model = # YOUR CODE HERE

In [ ]:
# Select the training arguments
training_args = TrainingArguments(
    output_dir="tinydnabert-finetuned",
    # YOUR CODE HERE: TRAIN THE MODEL FOR 3 EPOCHS
    # YOUR CODE HERE: SET THE BATCH SIZE PER DEVICE TO 16
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    # YOUR CODE HERE: SET THE LEARNING RATE TO 2E-5
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
)

In [ ]:
# Set up the trainer with the training arguments above
trainer = # YOUR CODE HERE

In [ ]:
# Fine-tune the model
# YOUR CODE HERE

#### Evaluate on promoter classification

In [ ]:
# TrainingArguments for evaluation only
training_args = TrainingArguments(
    output_dir="./results",
    # YOUR CODE HERE: SET THE BATCH SIZE PER DEVICE TO 24
    # YOUR CODE HERE: SET UP SO THAT WE DO EVALUATION BUT NOT TRAINING
    logging_dir="./logs",
    report_to="none",
)

In [ ]:
# Define metrics function
def compute_metrics(pred):
    labels = pred.label_ids
    # Handle output as tuple (logits,) or ModelOutput
    logits = pred.predictions[0] if isinstance(pred.predictions, tuple) else pred.predictions
    preds = logits.argmax(-1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average="macro")
    probabilities = softmax(logits, axis=-1)[:, 1] # probability for class 1
    auroc = roc_auc_score(labels, probabilities)  # using sklearn
    return {"accuracy": acc, "auroc": auroc, "f1": f1}

In [ ]:
# Set up the trainer with the training arguments above
# YOUR CODE HERE

In [ ]:
# Evaluate model
eval_results = # YOUR CODE HERE
print(eval_results)